# Tutorial 0: Camera Calibration with Anipose

**Pipeline Stage:** Producing the `calibration.toml` that 3D triangulation depends on

---

## Why This Tutorial Exists

Tutorials 1–3 take you from raw multi-camera video all the way to 3D skeletons.
But **Tutorial 3 (3D Triangulation) assumes you already have a `calibration.toml`**
for your camera rig — and never shows you how to make one.

This tutorial fills that gap. It takes you from **raw calibration videos** (one per
camera, all recording the same moving calibration board) to a finished
**`calibration.toml`**, using [Anipose](https://github.com/lambdaloop/anipose) /
[`sleap-anipose`](https://github.com/talmolab/sleap-anipose).

```
┌──────────────────────┐     ┌──────────────────────┐     ┌──────────────────────┐
│  Tutorial 0 (HERE)   │ ──► │  Tutorials 1 & 2     │ ──► │  Tutorial 3          │
│  Camera Calibration  │     │  2D Pose + ReID       │     │  3D Triangulation    │
│  → calibration.toml  │     │                       │     │  (uses calibration)  │
└──────────────────────┘     └──────────────────────┘     └──────────────────────┘
```

## What Calibration Actually Solves

To triangulate a 2D point seen in several cameras into a single 3D point, you must
know, for every camera:

1. **Intrinsics** — focal length, principal point, and lens distortion. *How does a
   3D point in front of this camera land on a pixel?*
2. **Extrinsics** — rotation and translation. *Where is this camera sitting in the
   world, and which way is it pointing?*

Calibration recovers all of these at once by watching a **known object** — a
calibration board with a precisely known geometry — move through the shared field of
view of every camera. Anipose detects the board corners in each view and runs
**iterative bundle adjustment** to jointly solve for every camera's intrinsics and
extrinsics while minimizing **reprojection error** (the pixel gap between where a
corner *was* detected and where the solved model says it *should* be).

## What You Will Learn

1. **Installation** — git clone → conda env → `sleap-anipose`, end to end
2. **The calibration board** — what a ChArUco board is, and printing your own
3. **Recording a good calibration video** — the single biggest driver of quality
4. **The session folder format** — camera subfolders + `calibration_images/`
5. **Extracting frames** — video → `calibration_images/*.jpg` (keeping only frames
   where the board is visible)
6. **Running calibration** — one `slap.calibrate(...)` call → `calibration.toml`
7. **Reading `calibration.toml`** — what every field means
8. **Judging quality** — reprojection-error histogram + overlays, and what's "good"

---

## Part 0: Installation (from scratch)

Calibration uses `sleap-anipose`, which wraps the `aniposelib` calibration engine.
Below is a complete setup starting from nothing. You only need to do this **once**.

> If you already made the `multicam-pose` environment from the main `README.md`, you
> can reuse it — just make sure `sleap-anipose` is installed in it (see step 4).

### 1. Install a conda/mamba distribution (skip if you already have one)

If you don't have `conda`, install [Miniforge](https://github.com/conda-forge/miniforge)
(recommended — ships the fast `mamba` solver and the conda-forge channel by default).

### 2. Clone the repositories

```bash
# The tutorials repo (this folder lives inside it) and sleap-anipose for reference
git clone https://github.com/talmolab/sleap-anipose.git
git clone https://github.com/lambdaloop/anipose.git   # optional: docs & reference
```

You do **not** need to install from the clone — `sleap-anipose` is on PyPI — but the
clone gives you `docs/FOLDER_STRUCTURE.md` and the source to read.

### 3. Create and activate the conda environment

```bash
conda create -n sleap-anipose python=3.9 -y
conda activate sleap-anipose
```

### 4. Install the calibration stack

```bash
# Core calibration engine (pulls in aniposelib, opencv, numba, toml, imageio, etc.)
pip install sleap-anipose

# Notebook + plotting utilities used in this tutorial
pip install jupyter ipykernel matplotlib pandas
```

### 5. Register a Jupyter kernel so this notebook can find the env

```bash
python -m ipykernel install --user \
    --name sleap-anipose \
    --display-name "Python (sleap-anipose)"
```

Then, in Jupyter, pick the **"Python (sleap-anipose)"** kernel (top-right) before
running the cells below.

> **OpenCV / ArUco note:** `sleap-anipose` depends on `opencv-contrib` for the ArUco
> module. `pip install sleap-anipose` handles this. If you ever see
> `module 'cv2' has no attribute 'aruco'`, you have a plain `opencv-python` shadowing
> it — fix with:
> `pip uninstall -y opencv-python opencv-contrib-python && pip install opencv-contrib-python`.

In [ ]:
# ============================================================
# STEP 0: Verify the environment
# ============================================================
import sys, platform
print(f"Python:   {sys.version.split()[0]}  ({platform.system()})")

import cv2
print(f"OpenCV:   {cv2.__version__}")
assert hasattr(cv2, "aruco"), (
    "cv2.aruco missing — install opencv-contrib-python (see the note above)."
)

import numpy as np
import matplotlib
print(f"numpy:    {np.__version__}")
print(f"mpl:      {matplotlib.__version__}")

try:
    import sleap_anipose as slap
    import aniposelib
    print(f"sleap-anipose: OK  |  aniposelib: {getattr(aniposelib, '__version__', 'installed')}")
except Exception as e:
    print(f"WARNING: could not import sleap_anipose ({e}).")
    print("Re-check Part 0 and that you selected the 'Python (sleap-anipose)' kernel.")

---

## Part 1: The Calibration Board

Anipose supports **checkerboards**, **ArUco** boards, and **ChArUco** boards.
`sleap-anipose` standardizes on the **ChArUco** board, and so will we.

### Why ChArUco?

A ChArUco board is a chessboard with an ArUco marker inside every white square:

```
┌───┬───┬───┬───┐
│▪ ▪│███│▪ ▪│███│   ███  = black chessboard square
├───┼───┼───┼───┤   ▪ ▪  = ArUco marker (a unique binary tag)
│███│▪ ▪│███│▪ ▪│
├───┼───┼───┼───┤   • Chessboard corners → sub-pixel accurate positions
│▪ ▪│███│▪ ▪│███│   • ArUco tags         → identify WHICH corner is which,
└───┴───┴───┴───┘                          even when the board is partly cut off
```

The chessboard gives **precision**; the ArUco tags give **unique identity** for each
corner. That combination means the board still calibrates correctly even when it's
tilted, partly out of frame, or only partly overlapping between two cameras — which is
exactly what happens when you wave it around a multi-camera volume.

### Board parameters

A ChArUco board is fully described by six numbers. These **must match your physical
board** — most importantly the lengths, which set the real-world **units** of your
entire 3D reconstruction.

| Parameter | Meaning | Example |
|---|---|---|
| `board_x` | squares across the width | `8` |
| `board_y` | squares down the height | `11` |
| `square_length` | chessboard square edge, **in your chosen units** | `24.0` (mm) |
| `marker_length` | ArUco marker edge, same units | `18.75` (mm) |
| `marker_bits` | bits per ArUco marker (4/5/6/7) | `4` |
| `dict_size` | ArUco dictionary size (50/100/250/1000) | `1000` |

> **Units set the world scale.** If `square_length` is in millimetres, every 3D
> coordinate you triangulate later (Tutorial 3) will be in millimetres. Measure your
> printed board's square edge with calipers and put the *real* number here.

In [ ]:
# ============================================================
# STEP 1: Define the calibration board
# ============================================================
# EDIT THESE to match YOUR physical board.
BOARD = {
    "board_x": 8,            # squares across (width)
    "board_y": 11,           # squares down (height)
    "square_length": 24.0,   # square edge length -> sets world units (mm here)
    "marker_length": 18.75,  # ArUco marker edge length (same units)
    "marker_bits": 4,        # 4x4 markers
    "dict_size": 1000,       # DICT_4X4_1000
}

# Persist the board spec next to your data so calibration is reproducible.
# This writes a board.toml that slap.calibrate() can also read directly.
import os
PROJECT_DIR = "calibration_demo"            # working directory for this tutorial
os.makedirs(PROJECT_DIR, exist_ok=True)
board_toml = os.path.join(PROJECT_DIR, "board.toml")

slap.write_board(board_name=board_toml, **BOARD)
print(f"Wrote board spec -> {board_toml}")
print(open(board_toml).read())

### Print your own board

You need a **physical** copy of the board to record calibration video. `sleap-anipose`
can draw a printable one for you. Print it at 100% scale (no "fit to page"), mount it
on something **rigid and flat** (foam board, clipboard), then **re-measure** the actual
printed square size and update `square_length` / `marker_length` above if it drifted.

In [ ]:
# ============================================================
# STEP 2: Draw a printable ChArUco board
# ============================================================
board_png = os.path.join(PROJECT_DIR, "charuco_board.png")

slap.draw_board(
    board_name=board_png,
    board_x=BOARD["board_x"],
    board_y=BOARD["board_y"],
    square_length=BOARD["square_length"],
    marker_length=BOARD["marker_length"],
    marker_bits=BOARD["marker_bits"],
    dict_size=BOARD["dict_size"],
    img_width=1440,
    img_height=1980,      # ~ width * board_y / board_x keeps squares square
    save="",              # (optional) path to also dump a board.toml
)

import matplotlib.pyplot as plt
img = plt.imread(board_png)
plt.figure(figsize=(6, 8))
plt.imshow(img, cmap="gray")
plt.title("Printable ChArUco board\n(print at 100% scale, mount flat & rigid)")
plt.axis("off")
plt.show()
print(f"Saved printable board -> {board_png}")

---

## Part 2: Recording a Good Calibration Video

**This is the step that decides your calibration quality.** The math is only as good as
the board coverage you feed it. Record one synchronized clip per camera of the board
being moved slowly through the capture volume.

### The rules that matter

- **Every camera must see the board a lot.** Bundle adjustment can only relate two
  cameras through frames where *both* see the board. Move so that overlapping pairs get
  many shared views.
- **Fill the whole volume.** Walk the board through the entire 3D space your subjects
  will occupy — near/far, left/right, floor/height. Corners of the volume matter most.
- **Tilt and rotate the board.** Vary its angle (pitch/yaw/roll), not just its
  position. Head-on-only views make focal length and distortion poorly constrained.
- **Move slowly / pause.** Rolling-shutter motion blur ruins corner detection. Glide,
  and hold briefly at each pose. Good lighting, no glare on the board.
- **Keep it flat and rigid.** Any bend in the board violates the known geometry and
  poisons the solve.
- **Enough frames.** After keeping only frames where the board is clearly visible, aim
  for **~100–300 good frames per camera**. A 1–3 minute clip usually gets you there.

### Synchronization

The cameras don't need microsecond sync for calibration (the board is roughly static
during each slow pose), but roughly aligned clips help. If your rig hardware-syncs for
the real recordings, just reuse that.

```
        Move the board through the WHOLE volume, tilting as you go:

           near ─────────────────────────► far
            ┌───────────────────────────────┐
            │   ◹      ◺       ◹      ◺      │   each ◹/◺ = board at a
        top │        ◺      every height &  │        different pose
            │   ◺       depth, tilted        │        (position + angle)
     bottom │      ◹        ◺        ◹       │
            └───────────────────────────────┘
```

---

## Part 3: The Session Folder Format

`sleap-anipose` is organized around a **session** folder. Each camera/view gets a
subfolder, and the board images/videos for that view live in a `calibration_images/`
subfolder inside it. `calibrate()` writes `calibration.toml` at the **session root**.

### Target structure (what we're building toward)

```
session/                              ← the "session" you pass to calibrate()
├── calibration.toml                  ← OUTPUT (what this tutorial produces)
├── calibration_metadata.h5           ← OUTPUT (detections + reprojections)
├── reprojection_histogram.png        ← OUTPUT (quality plot)
├── CAM1/
│   └── calibration_images/
│       ├── CAM1_frame_0000.jpg        ← extracted board frames (Part 4)
│       ├── CAM1_frame_0001.jpg
│       └── ...
├── CAM2/
│   └── calibration_images/
│       └── ...
├── ...
└── CAM6/
    └── calibration_images/
        └── ...
```

> **How `calibrate()` finds the video:** for each camera folder it globs `*/*.MOV`
> inside it. If it finds no video, it **builds one automatically** from the
> `calibration_images/*.jpg` frames. So our job in Part 4 is simply to fill each
> `calibration_images/` folder with good frames — `calibrate()` does the rest.

### Starting point

We assume you have one raw calibration video per camera. Point `RAW_CALIB_VIDEOS` at
them. The view name is the **dict key** and becomes the camera folder name (and the
camera's name inside `calibration.toml`).

In [ ]:
# ============================================================
# STEP 3: Point at your raw calibration videos (one per camera)
# ============================================================
from pathlib import Path

# EDIT: map each view/camera name -> its raw calibration video.
# Video names in calibration.toml will be exactly these keys.
RAW_CALIB_VIDEOS = {
    "CAM1": "/path/to/calibration/CAM1_calib.mp4",
    "CAM2": "/path/to/calibration/CAM2_calib.mp4",
    "CAM3": "/path/to/calibration/CAM3_calib.mp4",
    "CAM4": "/path/to/calibration/CAM4_calib.mp4",
    "CAM5": "/path/to/calibration/CAM5_calib.mp4",
    "CAM6": "/path/to/calibration/CAM6_calib.mp4",
}

# The session folder we will build and then calibrate.
SESSION = Path(PROJECT_DIR) / "session"
SESSION.mkdir(parents=True, exist_ok=True)

print(f"Session folder: {SESSION.resolve()}\n")
missing = []
for view, vid in RAW_CALIB_VIDEOS.items():
    ok = Path(vid).exists()
    print(f"  {view:6s}  {'OK ' if ok else 'MISSING'}  {vid}")
    if not ok:
        missing.append(view)
if missing:
    print(f"\n(!) Update the paths above for: {', '.join(missing)}")

---

## Part 4: Extract Board Frames → `calibration_images/`

Now we convert each raw video into `session/<view>/calibration_images/*.jpg`.

We don't keep *every* frame — most frames add nothing (board static, blurry, or out of
view) and just slow calibration down. Instead we **detect the ChArUco board** in each
frame and keep only frames with enough visible markers, sampled at a fixed stride so we
don't over-collect from a single pose.

The helper below is deliberately defensive about OpenCV versions: the `cv2.aruco` API
changed between OpenCV 4.6 and 4.7+, so we detect the API at runtime.

In [ ]:
# ============================================================
# STEP 4a: ChArUco detector (handles old & new cv2.aruco APIs)
# ============================================================
import cv2
from cv2 import aruco

_ARUCO_DICTS = {
    (4, 50): aruco.DICT_4X4_50,   (4, 100): aruco.DICT_4X4_100,
    (4, 250): aruco.DICT_4X4_250, (4, 1000): aruco.DICT_4X4_1000,
    (5, 50): aruco.DICT_5X5_50,   (5, 100): aruco.DICT_5X5_100,
    (5, 250): aruco.DICT_5X5_250, (5, 1000): aruco.DICT_5X5_1000,
    (6, 50): aruco.DICT_6X6_50,   (6, 100): aruco.DICT_6X6_100,
    (6, 250): aruco.DICT_6X6_250, (6, 1000): aruco.DICT_6X6_1000,
    (7, 50): aruco.DICT_7X7_50,   (7, 100): aruco.DICT_7X7_100,
    (7, 250): aruco.DICT_7X7_250, (7, 1000): aruco.DICT_7X7_1000,
}

def make_detector(board=BOARD):
    'Return (detect_fn, n_total_markers). detect_fn(gray) -> n_markers_seen.'
    dict_id = _ARUCO_DICTS[(board["marker_bits"], board["dict_size"])]
    aruco_dict = aruco.getPredefinedDictionary(dict_id)

    # New API (OpenCV >= 4.7): ArucoDetector object
    if hasattr(aruco, "ArucoDetector"):
        params = aruco.DetectorParameters()
        det = aruco.ArucoDetector(aruco_dict, params)
        def detect(gray):
            corners, ids, _ = det.detectMarkers(gray)
            return 0 if ids is None else len(ids)
    # Old API (OpenCV <= 4.6): free functions
    else:
        params = aruco.DetectorParameters_create()
        def detect(gray):
            corners, ids, _ = aruco.detectMarkers(gray, aruco_dict, parameters=params)
            return 0 if ids is None else len(ids)

    # A ChArUco board of (bx, by) has floor(bx*by/2) markers.
    n_markers = (board["board_x"] * board["board_y"]) // 2
    return detect, n_markers

_detect_markers, N_MARKERS = make_detector()
print(f"Detector ready. Full board shows up to {N_MARKERS} ArUco markers.")

In [ ]:
# ============================================================
# STEP 4b: Extract good board frames from each camera's video
# ============================================================
import shutil

# Tuning knobs:
FRAME_STRIDE      = 5      # examine every Nth frame (speed vs. coverage)
MIN_MARKER_FRAC   = 0.25   # keep frame if >= this fraction of markers are visible
MAX_FRAMES_PER_CAM = 250   # cap kept frames per camera (plenty for calibration)

min_markers = max(4, int(MIN_MARKER_FRAC * N_MARKERS))
print(f"Keeping frames with >= {min_markers} markers "
      f"(<= {MAX_FRAMES_PER_CAM} per camera, stride {FRAME_STRIDE}).\n")

def extract_view(view, video_path):
    out_dir = SESSION / view / "calibration_images"
    if out_dir.exists():
        shutil.rmtree(out_dir)         # start clean so re-runs are reproducible
    out_dir.mkdir(parents=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  {view}: could NOT open {video_path}")
        return 0

    kept, idx = 0, 0
    while kept < MAX_FRAMES_PER_CAM:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % FRAME_STRIDE == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if _detect_markers(gray) >= min_markers:
                # zero-padded sequential index -> stable ordering for the auto-built MOV
                fname = out_dir / f"{view}_frame_{kept:04d}.jpg"
                cv2.imwrite(str(fname), frame)
                kept += 1
        idx += 1
    cap.release()
    print(f"  {view}: scanned {idx} frames, kept {kept} board frames -> {out_dir}")
    return kept

counts = {}
for view, vid in RAW_CALIB_VIDEOS.items():
    if Path(vid).exists():
        counts[view] = extract_view(view, vid)
    else:
        print(f"  {view}: SKIPPED (video path not found)")

print("\nKept frames per camera:", counts)
if counts and min(counts.values()) < 50:
    print("(!) Some cameras have < 50 good frames. Consider re-recording that view, "
          "lowering FRAME_STRIDE, or MIN_MARKER_FRAC.")

In [ ]:
# ============================================================
# STEP 4c: Sanity-check one detection + show the folder tree
# ============================================================
# Draw detected markers on the first kept frame of the first camera.
first_view = next(iter(counts)) if counts else None
if first_view:
    sample = sorted((SESSION / first_view / "calibration_images").glob("*.jpg"))[0]
    img = cv2.imread(str(sample))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    dict_id = _ARUCO_DICTS[(BOARD["marker_bits"], BOARD["dict_size"])]
    aruco_dict = aruco.getPredefinedDictionary(dict_id)
    if hasattr(aruco, "ArucoDetector"):
        det = aruco.ArucoDetector(aruco_dict, aruco.DetectorParameters())
        corners, ids, _ = det.detectMarkers(gray)
    else:
        corners, ids, _ = aruco.detectMarkers(
            gray, aruco_dict, parameters=aruco.DetectorParameters_create())
    aruco.drawDetectedMarkers(img, corners, ids)

    plt.figure(figsize=(9, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{first_view}: {0 if ids is None else len(ids)} markers detected "
              f"in {sample.name}")
    plt.axis("off")
    plt.show()

# Print the resulting tree.
print("\nSession structure:")
for view in sorted(counts):
    imgs = sorted((SESSION / view / "calibration_images").glob("*.jpg"))
    print(f"{view}/  calibration_images/  ({len(imgs)} jpgs) "
          f"e.g. {imgs[0].name if imgs else '—'}")

---

## Part 5: Run Calibration → `calibration.toml`

Everything is now in the format `slap.calibrate()` expects. A single call:

1. Discovers the camera folders (each subfolder of the session is one view).
2. Builds a `.MOV` per camera from its `calibration_images/*.jpg` (since none exists).
3. Detects ChArUco corners across all views and runs **iterative bundle adjustment**.
4. Writes `calibration.toml` (the deliverable) plus optional QC outputs.

### Key arguments

| Argument | What it does |
|---|---|
| `session` | Path to the session folder (its subfolders are the views) |
| `board` | Board spec — a dict, a `CharucoBoard`, or the `board.toml` path |
| `calib_fname` | Where to write `calibration.toml` (**the main output**) |
| `metadata_fname` | `.h5` of detections + triangulations + reprojections (QC) |
| `histogram_path` | `.png` reprojection-error histogram (QC) |
| `reproj_path` | Folder to write detected-vs-reprojected overlay images (QC) |
| `excluded_views` | View **names** to leave out (e.g. a broken camera) |

> Calibration is CPU-heavy and can take a few minutes for 6 cameras × hundreds of
> frames. That's normal.

In [ ]:
# ============================================================
# STEP 5: Calibrate the session
# ============================================================
calib_toml   = SESSION / "calibration.toml"
metadata_h5  = SESSION / "calibration_metadata.h5"
histogram_png = SESSION / "reprojection_histogram.png"

cgroup, metadata = slap.calibrate(
    session=str(SESSION),
    board=BOARD,                       # dict is fine; could also pass board_toml
    excluded_views=(),                 # e.g. ("CAM5",) to drop a bad camera
    calib_fname=str(calib_toml),       # <-- produces calibration.toml
    metadata_fname=str(metadata_h5),
    histogram_path=str(histogram_png),
    reproj_path=str(SESSION),          # writes reprojection-*.png into each view
)

frames, detections, triangulations, reprojections = metadata
print(f"\nDone. Calibrated {len(cgroup.get_names())} cameras: {cgroup.get_names()}")
print(f"Common board frames used across all views: {len(frames)}")
print(f"calibration.toml -> {calib_toml.resolve()}")

---

## Part 6: Reading `calibration.toml`

The output is a plain TOML file with one `[cam_...]` block per camera. This is exactly
the file **Tutorial 3** loads to triangulate.

```toml
[cam_0]
name = "CAM1"
size = [1920, 1080]                                 # image resolution (px)
matrix = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]      # intrinsics
distortions = [k1, k2, p1, p2, k3]                  # lens distortion
rotation = [rx, ry, rz]                             # extrinsics (Rodrigues rvec)
translation = [tx, ty, tz]                          # extrinsics (in board units)
```

- **`matrix`** — intrinsic matrix. `fx, fy` focal lengths (px); `cx, cy` principal point.
- **`distortions`** — radial (`k1,k2,k3`) + tangential (`p1,p2`) lens distortion.
- **`rotation` / `translation`** — where the camera sits in the shared world frame.
  `translation` is in **your board units** (mm if you used mm), which is why the board
  measurement sets your 3D scale.

In [ ]:
# ============================================================
# STEP 6: Print and parse calibration.toml
# ============================================================
import toml

print(calib_toml.read_text()[:2000])
print("..." if calib_toml.stat().st_size > 2000 else "")

calib = toml.load(calib_toml)
print("\nParsed per-camera summary:")
for key, cam in calib.items():
    if key == "metadata":
        continue
    mat = np.array(cam["matrix"])
    print(f"  {cam.get('name', key):6s}  size={cam['size']}  "
          f"fx={mat[0,0]:7.1f}  fy={mat[1,1]:7.1f}  "
          f"cx={mat[0,2]:6.1f}  cy={mat[1,2]:6.1f}  "
          f"|t|={np.linalg.norm(cam['translation']):.1f}")

---

## Part 7: Judging Calibration Quality

**Never trust a calibration you haven't checked.** The single best metric is
**reprojection error**: triangulate the detected board corners back to 3D, project them
into every camera, and measure the pixel distance from the original detections.

### Rules of thumb

| Mean reprojection error | Verdict |
|---|---|
| **< 1 px** | Excellent |
| **1–3 px** | Good — fine for most 3D pose work |
| **3–5 px** | Marginal — usable but consider re-recording |
| **> 5 px** | Poor — re-record with better board coverage, or exclude a bad view |

If one camera is dragging the error up, re-run with that view in `excluded_views`, or
re-record calibration video for it (usually it never shared enough board views with the
others).

In [ ]:
# ============================================================
# STEP 7a: Reprojection-error histogram + per-camera breakdown
# ============================================================
# detections / reprojections: (n_cams, n_frames, n_corners, 2)
err = np.linalg.norm(detections - reprojections, axis=-1)   # (n_cams, n_frames, n_corners)
per_cam = np.nanmean(err.reshape(err.shape[0], -1), axis=1)

print("Per-camera mean reprojection error (px):")
for name, e in zip(cgroup.get_names(), per_cam):
    flag = "  <-- check this view" if e > 5 else ""
    print(f"  {name:6s}  {e:5.2f} px{flag}")
print(f"\nOverall mean: {np.nanmean(err):.2f} px   median: {np.nanmedian(err):.2f} px")

plt.figure(figsize=(8, 5))
plt.hist(err.ravel()[~np.isnan(err.ravel())], bins=np.linspace(0, 15, 60), density=True)
plt.axvline(np.nanmean(err), color="r", ls="--", label=f"mean {np.nanmean(err):.2f}px")
plt.xlabel("Reprojection error (px)")
plt.ylabel("PDF")
plt.title("Reprojection error across all views")
plt.legend()
plt.show()

# The saved histogram from calibrate():
if histogram_png.exists():
    print(f"\nSaved histogram -> {histogram_png}")

In [ ]:
# ============================================================
# STEP 7b: Look at a saved detection-vs-reprojection overlay
# ============================================================
# calibrate(reproj_path=...) drops reprojection-*.png into each view folder.
# Red '+' = detected corners, green 'x' = reprojected corners. They should overlap.
overlays = sorted(SESSION.glob("*/reprojection-*.png"))
if overlays:
    show = overlays[:min(3, len(overlays))]
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 6))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, show):
        ax.imshow(plt.imread(p))
        ax.set_title(f"{p.parent.name}/{p.name}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No reprojection overlays found (pass reproj_path= to calibrate to generate them).")

---

## Summary & Handoff to Tutorial 3

You went from **raw calibration videos → `calibration.toml`**:

```
raw per-camera calibration videos
        │  Part 4: detect board, extract frames
        ▼
session/<CAM>/calibration_images/*.jpg      ← sleap-anipose session format
        │  Part 5: slap.calibrate(...)
        ▼
session/calibration.toml                     ← the deliverable
   + calibration_metadata.h5, reprojection_histogram.png   (QC)
```

### Using it in the pipeline

Drop `calibration.toml` at the root of your **triangulation** session (the folder with
`cam1/ … cam6/` pose `.analysis.h5` files in Tutorial 3) and point the triangulation
step at it:

```python
import sleap_anipose as slap
slap.triangulate(
    p2d="/path/to/triangulation_session",
    calib="/path/to/triangulation_session/calibration.toml",   # <-- from THIS tutorial
    fname="points3d.h5",
)
```

Because `translation` in `calibration.toml` is in the **units of your board's
`square_length`**, your 3D coordinates in Tutorial 3 come out in those same units.

### Checklist for a good calibration

- [ ] Board printed at 100% scale, flat & rigid; `square_length`/`marker_length` measured
- [ ] Each camera has **100–300** good board frames after extraction
- [ ] Board was moved through the **whole volume**, tilted at many angles
- [ ] Overlapping camera pairs share **many** board views
- [ ] Mean reprojection error **< 3 px**; no single camera is an outlier
- [ ] `calibration.toml` copied to the triangulation session for Tutorial 3

### Troubleshooting

| Symptom | Likely fix |
|---|---|
| `cv2.aruco` missing | `pip install opencv-contrib-python` (remove plain `opencv-python`) |
| Few/no frames kept | Lower `MIN_MARKER_FRAC` / `FRAME_STRIDE`; check lighting & focus |
| High error on one camera | Add it to `excluded_views`, or re-record that view |
| High error everywhere | Board bent, wrong `square_length`, or too little volume coverage |
| Wrong 3D scale later | `square_length` didn't match the real printed board |